# Applio CEVC — Generator Registry

Этот блокнот запускает обычный интерфейс Applio из исследовательской ветки CEVC.

На этом этапе звук и формат старых моделей не должны измениться: выбор генератора вынесен в отдельный registry, а стандартный `HiFi-GAN` остаётся прежним.

Порядок работы:

1. При необходимости подключи Google Drive.
2. Установи Applio.
3. Подключи модели из `MyDrive/ApplioBackup`.
4. Запусти сервер и сделай обычную конвертацию.
5. Нажми **«Скачать лог»** и отправь ZIP для проверки.

Тестовые ячейки в блокнот не встроены. Проверки выполняются отдельно в CI.


In [ ]:
# @title 1. Подключить Google Drive (необязательно)
from google.colab import drive
from google.colab._message import MessageError

try:
    drive.mount("/content/drive")
except MessageError:
    print("❌ Не удалось подключить Google Drive.")


In [ ]:
# @title 2. Установить Applio CEVC
from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import shutil
import subprocess
import sys

APP_DIR = Path("/content/Applio")
DIAG_DIR = APP_DIR / "cevc_diagnostics"
INSTALL_LOG = DIAG_DIR / "install.log"
ENVIRONMENT_JSON = DIAG_DIR / "environment.json"
REPO_URL = "https://github.com/egor125552/Applio.git"
REPO_REF = "agent/compact-expressive-vc-architecture"
NOTEBOOK_BUILD = "cevc-registry-v2"


def run_logged(command, *, cwd=None, check=True, env=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    INSTALL_LOG.parent.mkdir(parents=True, exist_ok=True)
    with INSTALL_LOG.open("a", encoding="utf-8") as log_file:
        log_file.write(f"\n$ {' '.join(command)}\n")
        log_file.write(completed.stdout or "")
        log_file.write(f"\n[exit_code={completed.returncode}]\n")
    if completed.stdout:
        print(completed.stdout, flush=True)
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Команда завершилась с кодом {completed.returncode}: {' '.join(command)}"
        )
    return completed


if APP_DIR.exists():
    shutil.rmtree(APP_DIR)

Path("/content").mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "config", "--global", "advice.detachedHead", "false"], check=False)
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_REF,
        REPO_URL,
        str(APP_DIR),
    ],
    check=True,
)
DIAG_DIR.mkdir(parents=True, exist_ok=True)
INSTALL_LOG.write_text(
    f"CEVC notebook build: {NOTEBOOK_BUILD}\n"
    f"Started UTC: {datetime.now(timezone.utc).isoformat()}\n",
    encoding="utf-8",
)

run_logged(["apt-get", "update", "-y"])
run_logged(["apt-get", "install", "-y", "portaudio19-dev"])
run_logged(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"])

uv_path = shutil.which("uv") or "/root/.local/bin/uv"
run_logged(
    [
        uv_path,
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu128",
        "--index-strategy",
        "unsafe-best-match",
    ],
    cwd=APP_DIR,
)
run_logged(
    [uv_path, "pip", "install", "-q", "ngrok", "jupyter-ui-poll", "ipywidgets"],
    cwd=APP_DIR,
)
run_logged(["npm", "install", "-g", "-q", "localtunnel"], cwd=APP_DIR)
run_logged(
    [
        sys.executable,
        "core.py",
        "prerequisites",
        "--models",
        "True",
        "--pretraineds_hifigan",
        "True",
    ],
    cwd=APP_DIR,
)

commit = run_logged(
    ["git", "rev-parse", "HEAD"], cwd=APP_DIR, check=False
).stdout.strip()

environment = {
    "notebook_build": NOTEBOOK_BUILD,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL,
    "ref": REPO_REF,
    "commit": commit,
    "python": sys.version,
    "platform": platform.platform(),
}
try:
    import torch

    environment.update(
        {
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda,
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        }
    )
except Exception as error:
    environment["torch_error"] = repr(error)

ENVIRONMENT_JSON.write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("✅ Установка завершена.")
print(f"Ветка: {REPO_REF}")
print(f"Коммит: {commit}")


In [ ]:
# @title 3. Подключить модели из Google Drive (необязательно)
from pathlib import Path

APP_DIR = Path("/content/Applio")
LOCAL_LOGS = APP_DIR / "logs"
DRIVE_BACKUP = Path("/content/drive/MyDrive/ApplioBackup")

LOCAL_LOGS.mkdir(parents=True, exist_ok=True)

if Path("/content/drive").is_mount():
    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    linked = 0
    for item in DRIVE_BACKUP.iterdir():
        target = LOCAL_LOGS / item.name
        if target.exists() or target.is_symlink():
            continue
        target.symlink_to(item, target_is_directory=item.is_dir())
        linked += 1
    print(f"✅ Подключено объектов из Drive: {linked}")
    print(f"Папка моделей: {DRIVE_BACKUP}")
else:
    print("Google Drive не подключён. Можно загрузить модель через интерфейс Applio.")


In [ ]:
# @title 4. Запустить Applio и включить логирование
from datetime import datetime, timezone
from google.colab import files, output
from IPython.display import HTML, display
from ipywidgets import Button, Output, VBox
from pathlib import Path
import json
import os
import queue
import re
import signal
import socket
import subprocess
import sys
import threading
import time
import zipfile

method = "gradio"  # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = ""  # @param {type:"string"}
startup_timeout = 75  # @param {type:"integer"}

APP_DIR = Path("/content/Applio")
DIAG_DIR = APP_DIR / "cevc_diagnostics"
APP_LOG = DIAG_DIR / "app.log"
GPU_CSV = DIAG_DIR / "gpu_usage.csv"
PIDS_JSON = DIAG_DIR / "processes.json"
DIAG_DIR.mkdir(parents=True, exist_ok=True)


def stop_previous_processes():
    if PIDS_JSON.exists():
        try:
            previous = json.loads(PIDS_JSON.read_text(encoding="utf-8"))
            for pid in previous.get("pids", []):
                try:
                    os.killpg(int(pid), signal.SIGTERM)
                except (ProcessLookupError, PermissionError, ValueError):
                    pass
        except Exception:
            pass
    for old_pid_file in (APP_DIR / "assets").glob("*_pid.txt"):
        try:
            old_pid_file.unlink()
        except OSError:
            pass


def gpu_monitor(stop_event):
    GPU_CSV.write_text(
        "timestamp_utc,index,name,memory_used_mib,memory_total_mib,utilization_gpu_percent\n",
        encoding="utf-8",
    )
    query = "index,name,memory.used,memory.total,utilization.gpu"
    while not stop_event.is_set():
        completed = subprocess.run(
            [
                "nvidia-smi",
                f"--query-gpu={query}",
                "--format=csv,noheader,nounits",
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        timestamp = datetime.now(timezone.utc).isoformat()
        if completed.returncode == 0:
            with GPU_CSV.open("a", encoding="utf-8") as handle:
                for row in completed.stdout.splitlines():
                    handle.write(f"{timestamp},{row}\n")
        stop_event.wait(0.5)


def collect_model_manifest():
    import torch

    records = []
    logs_dir = APP_DIR / "logs"
    for root, directories, filenames in os.walk(logs_dir, followlinks=True):
        directories[:] = [name for name in directories if name != ".ipynb_checkpoints"]
        for filename in filenames:
            path = Path(root) / filename
            if path.suffix.lower() not in {".pth", ".onnx", ".index"}:
                continue
            record = {
                "path": str(path.relative_to(APP_DIR)),
                "resolved_path": str(path.resolve()),
                "suffix": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            }
            if path.suffix.lower() == ".pth":
                try:
                    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
                    weights = (
                        checkpoint.get("weight", checkpoint)
                        if isinstance(checkpoint, dict)
                        else checkpoint
                    )
                    if isinstance(weights, dict):
                        tensors = [
                            value for value in weights.values() if hasattr(value, "numel")
                        ]
                        record["parameter_count"] = int(
                            sum(value.numel() for value in tensors)
                        )
                        record["tensor_count"] = len(tensors)
                    if isinstance(checkpoint, dict):
                        for key in ("version", "vocoder", "f0", "sr"):
                            if key in checkpoint:
                                record[key] = checkpoint[key]
                except Exception as error:
                    record["inspection_error"] = repr(error)
            records.append(record)
    manifest = DIAG_DIR / "model_manifest.json"
    manifest.write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return manifest


def collect_process_snapshot():
    completed = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv,noheader,nounits",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    snapshot = DIAG_DIR / "gpu_processes.txt"
    snapshot.write_text(completed.stdout or "", encoding="utf-8")
    return snapshot


def build_log_bundle():
    collect_model_manifest()
    collect_process_snapshot()
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    archive = Path(f"/content/CEVC_registry_log_{timestamp}.zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(DIAG_DIR.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=path.relative_to(DIAG_DIR))
    return archive


def port_is_open(port=6969):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


stop_previous_processes()
APP_LOG.write_text("", encoding="utf-8")

stop_gpu_monitor = threading.Event()
threading.Thread(
    target=gpu_monitor,
    args=(stop_gpu_monitor,),
    daemon=True,
).start()

command = [sys.executable, "-u", "app.py", "--listen", "--client"]
if method == "gradio":
    command.append("--share")

process_env = os.environ.copy()
process_env["PYTHONUNBUFFERED"] = "1"
process_env["GRADIO_ANALYTICS_ENABLED"] = "False"

app_process = subprocess.Popen(
    command,
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=process_env,
    start_new_session=True,
)
pids = [app_process.pid]
public_url = None
line_queue = queue.Queue()


def pump_output(process):
    with APP_LOG.open("a", encoding="utf-8", buffering=1) as log_handle:
        for line in iter(process.stdout.readline, ""):
            log_handle.write(line)
            log_handle.flush()
            line_queue.put(line)
    process.stdout.close()


threading.Thread(target=pump_output, args=(app_process,), daemon=True).start()

if method == "localtunnel":
    tunnel_process = subprocess.Popen(
        ["lt", "--port", "6969"],
        cwd=APP_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=process_env,
        start_new_session=True,
    )
    pids.append(tunnel_process.pid)
    threading.Thread(target=pump_output, args=(tunnel_process,), daemon=True).start()
elif method == "ngrok":
    import ngrok

    listener = ngrok.forward(6969, authtoken=ngrok_token.strip())
    public_url = listener.url()

PIDS_JSON.write_text(
    json.dumps(
        {
            "started_at_utc": datetime.now(timezone.utc).isoformat(),
            "method": method,
            "pids": pids,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

patterns = [
    re.compile(r"https://[a-zA-Z0-9-]+\.gradio\.live"),
    re.compile(r"https://[^\s]+\.loca\.lt"),
]

print("Запускаю сервер…", flush=True)
print("Вывод теперь идёт сразу; молчания на три минуты быть не должно.", flush=True)

deadline = time.time() + max(20, int(startup_timeout))
last_status = 0.0
local_ready = False

while time.time() < deadline:
    while True:
        try:
            line = line_queue.get_nowait()
        except queue.Empty:
            break
        print(line, end="", flush=True)
        for pattern in patterns:
            match = pattern.search(line)
            if match:
                public_url = match.group(0)

    if app_process.poll() is not None:
        break

    if port_is_open():
        local_ready = True
        if public_url is None:
            try:
                public_url = output.eval_js("google.colab.kernel.proxyPort(6969)")
            except Exception:
                pass

    if public_url:
        break

    if time.time() - last_status >= 5:
        elapsed = int(max(0, startup_timeout - (deadline - time.time())))
        state = "порт 6969 уже открыт" if local_ready else "сервер ещё загружается"
        print(f"… {elapsed} с: {state}", flush=True)
        last_status = time.time()

    time.sleep(0.25)

while True:
    try:
        line = line_queue.get_nowait()
    except queue.Empty:
        break
    print(line, end="", flush=True)

if public_url:
    display(
        HTML(
            f'<p><b>Applio открыт:</b> '
            f'<a href="{public_url}" target="_blank">{public_url}</a></p>'
        )
    )
elif local_ready:
    print("✅ Сервер запущен на порту 6969, но публичная ссылка не создалась.")
    print("Перезапусти только эту ячейку с method = 'localtunnel'.")
else:
    print("❌ Сервер не открыл порт 6969.")
    if APP_LOG.exists():
        print("\nПоследние строки лога:")
        print("\n".join(APP_LOG.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()[-40:]))

download_output = Output()
download_button = Button(
    description="Скачать лог",
    button_style="primary",
    icon="download",
    tooltip="Собрать и скачать технический лог CEVC",
)


def download_log(_):
    with download_output:
        download_output.clear_output()
        try:
            archive = build_log_bundle()
            print(f"Готово: {archive.name}")
            files.download(str(archive))
        except Exception as error:
            print(f"Не удалось собрать лог: {error!r}")


download_button.on_click(download_log)
display(VBox([download_button, download_output]))

print("✅ Ячейка запуска завершена. Сервер продолжает работать в фоне.")
print("После конвертации нажми «Скачать лог».")
